# 04 — Aggregate scores

Rolls the per-branch intersections up to one row per tract.

| | |
|---|---|
| **Reads** | `results/coverage_details.parquet`, `clean/population_demographics.csv` |
| **Writes** | `results/coverage_scores.parquet`, `results/coverage_scores.csv` |
| **Needs** | `cpl-s3` |

Two different numbers come out of this, and the difference matters:

- **`raw_score`** — the overlaps added up. Two branches each reaching half a
  tract gives `1.0`, and three overlapping branches can push it past `1.0`.
  It is a measure of *service density*.
- **`percent_covered`** — the branch footprints merged first, then measured.
  Double coverage counts once, so this never exceeds `1.0`. It is a measure of
  *reach*.

### Running this as an Elyra node

Drag this notebook onto a pipeline canvas, then open its **node properties**:

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 (or any image with pip access) |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |

The notebook installs its own dependencies in the first cell, so any runtime
image works. Everything it reads and writes lives in S3 — nothing is passed
between nodes as a file, so node ordering is the only dependency.

In [ ]:
%pip install -q boto3 shapely pandas pyarrow pyproj

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("aggregate_scores")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Load

In [ ]:
from pyproj import Geod
from shapely import wkt
from shapely.ops import unary_union

geod = Geod(ellps="WGS84")

obj = s3.get_object(Bucket=BUCKET, Key="results/coverage_details.parquet")
coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
log.info("Loaded %d coverage records", len(coverage))

obj = s3.get_object(Bucket=BUCKET, Key="clean/population_demographics.csv")
tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))

tract_area_map = {
    row["geoid"]: abs(geod.geometry_area_perimeter(wkt.loads(row["geometry"]))[0])
    for _, row in tracts.iterrows()
}
log.info("Measured %d tracts", len(tract_area_map))

## Aggregate per tract / mode / travel time

In [ ]:
results = []
groups = coverage.groupby(["geoid", "mode", "travel_time"])

for i, ((geoid, mode, tt), group) in enumerate(groups):
    merged = unary_union([wkt.loads(w) for w in group["intersection_wkt"]])
    merged_area = abs(geod.geometry_area_perimeter(merged)[0])
    tract_area = tract_area_map.get(geoid, 1)

    results.append({
        "geoid": geoid,
        "mode": mode,
        "travel_time": tt,
        "raw_score": group["overlap_proportion"].sum(),
        "percent_covered": min(merged_area / tract_area if tract_area > 0 else 0, 1.0),
        "num_branches": len(group),
    })

    if (i + 1) % 500 == 0:
        log.info("Aggregated %d/%d groups", i + 1, len(groups))

scores = pd.DataFrame(results)
scores.head()

## Write

In [ ]:
buf = io.BytesIO()
scores.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="results/coverage_scores.parquet", Body=buf.getvalue())

buf = io.BytesIO()
scores.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="results/coverage_scores.csv", Body=buf.getvalue())

log.info("Wrote %d tract scores across %d modes", len(scores), scores["mode"].nunique())
scores.groupby(["mode", "travel_time"])["percent_covered"].describe()[["count", "mean", "max"]]